In [8]:
from ..Tools import *


load_dotenv(override=True)

DEEPSEEK_API_KEY=os.getenv('DEEPSEEK_API_KEY')
DEEPSEEK_BASE_URL=os.getenv('DEEPSEEK_BASE_URL')
model=init_chat_model(
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    model='deepseek-v4-flash',
    model_provider='deepseek'
    #extra_body={"thinking":{"type":"enabled"}
)

#如果文档说明中有参数Args说明，则方法参数中必须标注类型，否则工具调用会出错
#parse_docstring为是否解析docstring中的参数并填充，name_or_callable为更改工具名称(不是调用的方法名，而是tool_calls里显示的名字)
@tool(parse_docstring=True,name_or_callable='getweather')
def get_weather(city:str='乐山',time:Optional[datetime] =None) ->str:
    """
    获取指定城市指定时间的天气

    Args:
        city: 查询的城市
        time: 查询的时间(若未传自动获取当前时间)
    """
    if time is None:
        time = datetime.now()
    time=time.strftime('%Y-%m-%d ')
    return '时间'+time+'下'+city+"天气状况晴朗。"

In [5]:
#使用tool注解的情况不需要手动拼接出toolMessage结果
model_withtools=model.bind_tools([get_weather])
messages=[
    HumanMessage('2026-7-15天气如何')
]
response=model_withtools.invoke(messages)
messages.append(response)
response.pretty_print()
tool_calls=response.tool_calls
print(type(tool_calls))
for tool_call in tool_calls:
    if tool_call['name']=='get_weather':
        print(type(tool_call))
        print(tool_call)
        #被tool注解修饰的工具方法可以调用invoke方法接收方法入参并执行
        tool_message=get_weather.invoke(tool_call)
        messages.append(tool_message)
print("=====================> messages <=====================")
for msg in messages:
    print(msg,end='\n',flush=True)
print("=====================> messages <=====================")

================================== Ai Message ==================================

好的！我先查一下2026年7月15日乐山的天气情况。
Tool Calls:
  getweather (call_00_3G9jIKQVLd8HAzDOWO7T0499)
 Call ID: call_00_3G9jIKQVLd8HAzDOWO7T0499
  Args:
    city: 乐山
    time: 2026-07-15T00:00:00
<class 'list'>
=====================> messages <=====================
content='2026-7-15天气如何' additional_kwargs={} response_metadata={}
content='好的！我先查一下2026年7月15日乐山的天气情况。' additional_kwargs={'refusal': None, 'reasoning_content': '用户想知道2026年7月15日的天气，但没有指定城市。让我先获取当前时间，然后查询天气。不过，工具没有指定城市参数时默认是"乐山"，我先查询乐山2026年7月15日的天气。'} response_metadata={'token_usage': {'completion_tokens': 140, 'prompt_tokens': 336, 'total_tokens': 476, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 52, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 336}, 'model_provider': 'deepseek', '

In [11]:
convert_to_openai_tool(get_weather)

{'type': 'function',
 'function': {'name': 'getweather',
  'description': '获取指定城市指定时间的天气',
  'parameters': {'properties': {'city': {'default': '乐山',
     'description': '查询的城市',
     'type': 'string'},
    'time': {'anyOf': [{'format': 'date-time', 'type': 'string'},
      {'type': 'null'}],
     'default': None,
     'description': '查询的时间(若未传自动获取当前时间)'}},
   'type': 'object'}}}